In [1]:
import sys
sys.dont_write_bytecode = True
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import datetime as dt
from scipy.stats import pearsonr
import pickle
import torch
import torch.nn as nn
import os
import time
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import math
from math import sqrt
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
import codes.mnn_Utils as mnn
from codes.make_dataset import DatasetHist
import random

In [2]:
val_path = "../data/data_generated/val/val_ips_omni_df.csv"

In [3]:
val_dataset = DatasetHist(val_path)

Input features are ['dist', 'hla', 'hlo', 'gla', 'glo', 'carr', 'v', 'er', 'sc_indx', 'time', 'day_total', 'time_trgt', 'input']
Target features are ['swSpeed_Smth_0']
begins: 71994 ends: 77399


In [4]:
val_load = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [5]:
model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=0.2, device=None)

In [6]:
for i, val in enumerate(val_load):
    print(val[1].shape, val[2].shape, val[1].to(torch.float32).dtype)
    model.eval()
    output = model(val[1].to(torch.float32))
    # print(output)
    loss = F.mse_loss(val[2], output)
    print(loss.item())
    break

torch.Size([16, 13, 32]) torch.Size([16, 16]) torch.float32
0.005332596550666327


In [9]:
runCount = 0
device = torch.device("cuda") 
epochs = 30

startT = time.time()
lrs = [1.0e-3,1.0e-2] #learning rate
weight_decays = [1.0e-4,1.0e-3] #weight decay regularization
dropouts = [0.3,0.2,0.4]  #dropouts
batch_sizes = [32,16,64]      # batchsize
gammas = [0.95,0.90,0.85]#,0.80] #learning rate scheduler, reduces the learning rate as training gets cloes to the minima

run_count = 0

def start_training(train_set, val_set, train_steps, epochs):
    # Get data loaded
    train_load = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_load = DataLoader(val_set, batch_size=batch_size, shuffle=True)

    # Load model
    model = mnn.model_base(xCh=13, LEN=32, emDim=128, dropout=dropout, device=None)
    # Use GPU
    # model = model.to(device)

    # Inititialize optimizer ans scheduler
    optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer,step_size=1,gamma=gamma)

    def train(data_load=train_load, epoch, run_count=run_count, train_steps=train_steps):
        model.train()
        running_loss = 0.0
        running_loss_y = 0.0
        for i, data in enumerate(data_load):
            # zero the parameter gradients
            optimizer.zero_grad()
            loss, y_loss = doStep(data)
            del data

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            running_loss_y += y_loss.item()
            if i == train_steps:
                break

        # print statistics
        running_loss = running_loss/train_steps
        running_loss_y = running_loss_y/train_steps
        print('Run %d [Epoch %d] loss: %.4e' %(run_count, epoch, running_loss_y*800.0))
        return running_loss, running_loss_y

    def deStep(data):
        x = data[1].to(torch.float32)
        y = data[2]
        y_out = model(x)

        loss = F.mse_loss(y_out, y, reduction='none')
        loss = loss * y * 100.0 # Why this scaling
        loss = loss.mean()
        y_loss = F.mse_loss(outputs,y)

        return loss, y_loss

    def validate(val_steps, epoch, run_count):
        model.eval()
        with torch.no_grad():
            running_loss = 0.0
            running_loss_y = 0.0
            for i, data in enumerate(val_load):
                loss, y_loss = doStep(data)
                del data
    
                running_loss += loss.item()
                running_loss_y += y_loss.item()
                if i == val_steps:
                    break
    
            # print statistics
            running_loss = running_loss/val_steps
            running_loss_y = running_loss_y/val_steps
            print('Run %d [Epoch %d] val loss: %.4e' %(run_count, epoch, running_loss_y*800.0))
        return running_loss, running_loss_y
    